# MDEN: обучение с экспериментальным физическим regularizer
Основной ноутбук проекта, ревизия 2026-09-18. Сверка с `PINN.pdf`, Energy 339 (2025) 138953.

**Контракт:** 32 последовательных блока × 5 признаков → следующие 8 SOC и SOH; stride 8.
Это реконструкция архитектуры по статье и **адаптация данных KIT**, не воспроизведение MIT-эксперимента.
KIT SOC — опубликованная интервальная оценка, SOH — интерполированная разметка по ёмкости.
`cycle` в этом адаптере содержит EFC, а `time_s` — время эксперимента, не номер и время внутри отдельного цикла.

Установите из корня: `python -m pip install -e ".[notebook,test]"`. PyTorch с CUDA выбирайте для своей среды; эта команда не устанавливает CUDA-драйвер.
Перезапустите kernel и выполните все ячейки. Основная логика находится в модулях; ячейки разделяют параметры, действия и проверки.
Старые веса архитектуры и кеш v7 не совместимы с новой методикой: запускается новая генерация данных и новое обучение.

**GPU optimization update:** execution benchmark, asynchronous data prefetch, optional compiled scan and selection by raw CSV volume. See `docs/GPU_OPTIMIZATION_RU.md`. CUDA speed must be measured locally.

In [ ]:
import os
import sys
from pathlib import Path
PROJECT_ROOT = Path(os.getenv("MDEN_PROJECT_ROOT", ".")).expanduser().resolve()
if not (PROJECT_ROOT / "mden_battery").is_dir() and (PROJECT_ROOT.parent / "mden_battery").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "mden_battery").is_dir(), "Укажите MDEN_PROJECT_ROOT: корень распакованного архива"
sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import json
import time
from datetime import datetime, timezone
from dataclasses import asdict, fields
import numpy as np
import matplotlib.pyplot as plt
import polars as pl
import torch
import yaml
from tqdm.auto import tqdm
import mden_battery
print("Импортируется:", mden_battery.__file__)
assert Path(mden_battery.__file__).resolve().is_relative_to(PROJECT_ROOT), "Загружена старая установленная версия; перезапустите kernel"

In [ ]:
from mden_battery.model import MDEN, MDENConfig
from mden_battery.loss import MDENJointLoss
from mden_battery.physics import PhysicsConfig, diagnose_physics_targets
from mden_battery.cache import cleanup_prepared_cache, save_data_recipe
from mden_battery.training import seed_everything
from mden_battery.data.article_preprocessing import WindowConfig
from mden_battery.data.log_age import PrepConfig, INPUT_COLUMNS, ARRAY_COLUMNS, prepare_dataset, validate_prepared_dataset
from mden_battery.data.window_batches import WindowBatchDataset, choose_cache_device
from mden_battery.runtime import configure_precision, move_batch_to_device, build_optimizer, choose_micro_batch_size, train_epoch_amp, eval_epoch_amp
from mden_battery.checkpoints import TRACKED, new_training_state, complete_epoch, save_epoch, resume_training, load_inference
from mden_battery.visualization import plot_history, collect_examples, plot_examples
from mden_battery.performance import configure_execution, tune_training_setup

## 1. Данные и параметры
Поменяйте `RAW_DATA_DIR`, если данные лежат в другом месте. `DATASET_CELL_LIMIT` — единственное место выбора 10/30/50 батарей. Число split и имя кеша вычисляются автоматически. Физический аккумулятор определяется как P+номер повтора; S/C — его канал подключения.

In [ ]:
RAW_DATA_DIR = Path(os.getenv(
    "MDEN_RAW_DATA_DIR",
    "/home/jupyter/datasphere/Klim_PiNN/DataSet/battery_aging_log_data_v2/extracted/10.35097-kww7jv8ajuvchcah/data/dataset/cell_log_age",
)).expanduser()
DATASET_CELL_LIMIT = 10  # 30 -> 18/6/6; 50 -> 30/10/10
TARGET_RAW_GB = None  # 10.0 = около 10 ГБ выбранных исходных CSV
SEED = 42
REBUILD_PREPARED_DATA = False
CLEAN_OLD_PREPARED = True  # только узнаваемые поколения в PREPARED_ROOT
DELETE_PREPARED_AFTER_RUN = True  # после графиков закрыть datasets и удалить NPY
MAX_ANCHOR_GAP_DAYS = None  # например 28; порог требует проверки частоты измерений
SMOKE_TEST = os.getenv("MDEN_SMOKE_TEST", "0") == "1"
if SMOKE_TEST:
    TARGET_RAW_GB = None
    print("СИНТЕТИЧЕСКИЙ SMOKE TEST: результаты не оценивают качество на KIT")
assert RAW_DATA_DIR.is_dir(), RAW_DATA_DIR

In [ ]:
INPUT_LENGTH = 32
HORIZON = 8
WINDOW_STRIDE = 8
TARGET_INTERVAL_S = 30
dataset_label = (f"{DATASET_CELL_LIMIT}_cells" if TARGET_RAW_GB is None
                 else f"{TARGET_RAW_GB:g}GB")
PREPARED_ROOT = PROJECT_ROOT / "data" / f"prepared_log_age_30s_{dataset_label}_v8"
prep_config = PrepConfig(
    raw_dir=RAW_DATA_DIR, prepared_dir=PREPARED_ROOT, cell_limit=DATASET_CELL_LIMIT, target_raw_gb=TARGET_RAW_GB,
    selection_seed=SEED, split_seed=SEED, input_length=INPUT_LENGTH,
    horizon=HORIZON, stride=WINDOW_STRIDE, interval_s=TARGET_INTERVAL_S,
    soc_scale=100.0, rated_capacity_ah=3.0, show_progress=not SMOKE_TEST,
    max_anchor_gap_s=None if MAX_ANCHOR_GAP_DAYS is None else MAX_ANCHOR_GAP_DAYS * 86400,
)
print("Данные:", RAW_DATA_DIR)
print("Кеш:", PREPARED_ROOT)

In [ ]:
FIXED_MICRO_BATCH_SIZE = None  # например 512 или 4224; None = короткий GPU probe
BATCH_SIZE_CANDIDATES = (256, 512, 1024, 2048, 4096, 8192)
FEM_EXECUTION = "auto"  # измерить dynamic / static / matrix
COMPILE_SCAN = True  # сравнить compiled native scan с eager
AMP_DTYPE = "auto"  # FP16 на T4; BF16 при аппаратной поддержке
ALLOW_TF32 = False  # отдельный эксперимент, по умолчанию точный FP32
MAX_VRAM_FRACTION = 0.88
PROBE_WARMUP_STEPS = 3
PROBE_TIMED_STEPS = 10
GRADIENT_ACCUMULATION_STEPS = 1
USE_MIXED_PRECISION = True
SHOW_BATCH_PROGRESS = True
DATA_CACHE_DEVICE = "auto"  # auto / cpu / cuda
RESUME_PATH = None  # путь к last.pt новой версии; не старый checkpoint
if SMOKE_TEST:
    FIXED_MICRO_BATCH_SIZE = 8
    SHOW_BATCH_PROGRESS = False
    torch.set_num_threads(1)

In [ ]:
raw_config = yaml.safe_load((PROJECT_ROOT / "configs/article_mden.yaml").read_text())
unknown = set(raw_config["model"]) - {f.name for f in fields(MDENConfig)}
assert not unknown, f"Неизвестные поля YAML: {unknown}"
model_config = MDENConfig(**raw_config["model"])
train_config = raw_config["training"]
PHYSICS_ENABLED = True  # False = исходный MDEN loss для контрольного эксперимента
physics_config = PhysicsConfig(**raw_config.get("physics", {})) if PHYSICS_ENABLED else None
LOSS_CONFIG = {"physics": asdict(physics_config) if physics_config else None}
assert raw_config["data"]["input_columns"] == INPUT_COLUMNS
assert model_config.input_dim == 5 and model_config.horizon == HORIZON
assert (model_config.soc_depth, model_config.shared_depth, model_config.soh_depth) == (2, 4, 6)
assert model_config.conv_rounds == 4
assert not model_config.fem_residual and model_config.mamba_layout == "figure4"
print("✓ конфигурация соответствует выбранному прочтению формул и рисунков")
if physics_config is not None:
    assert physics_config.interval_s == prep_config.interval_s
    assert physics_config.rated_capacity_ah == prep_config.rated_capacity_ah


In [ ]:
LEARNING_RATE = float(train_config["learning_rate"])
WEIGHT_DECAY = float(train_config["weight_decay"])
NUM_EPOCHS = 1 if SMOKE_TEST else int(train_config["epochs"])  # 3 для первого прогона 10 ГБ
PATIENCE = int(train_config["patience"])
GRADIENT_CLIP_NORM = float(train_config["gradient_clip_norm"])
MONITOR = str(train_config.get("monitor", "loss"))  # loss / rmse_soc / rmse_soh / balanced_rmse
assert MONITOR in TRACKED.values()
print("LR:", LEARNING_RATE, "WD:", WEIGHT_DECAY, "monitor:", MONITOR)

## 2. Предобработка
1. Находим CSV; выбираем уникальные физические батареи с доступными измерениями ёмкости.
2. Делим батареи на train/val/test 60/20/20 до вычисления scaler.
3. Читаем нужные столбцы Polars. Проверяем 2-секундные точки внутри настоящих временных 30-секундных корзин; неполные/некорректные корзины исключаются. Усредняем V/I/T/EFC/SOC, согласуя с опубликованными 30-секундными файлами.
4. Переносим метку времени с левого края усреднения к правому — моменту доступности блока.
5. SOC = `soc_est/100`. SOH = линейная по времени интерполяция `cap_aged_est_Ah/3`. За пределами опорных измерений цели не выдумываем: такие строки исключаются.
6. Удаляем NaN/Inf и недопустимые SOC/SOH; разбиваем на непрерывные сегменты. Сегменты короче 40 точек исключаются.
7. Считаем mean/std только на train в float64, нормализуем 5 входов, сохраняем float32 NPY. SOC/SOH не стандартизуются.

Новая обработка не может восстановить разрывы, уже интерполированные поставщиком LOG_AGE. Настоящих границ циклов в используемых семи полях нет; EFC не округляется для создания фиктивных циклов.

In [ ]:
GENERATION_DIR, cache_reused = prepare_dataset(prep_config, rebuild=REBUILD_PREPARED_DATA)
if CLEAN_OLD_PREPARED:
    print("Очистка старого кеша:", cleanup_prepared_cache(PREPARED_ROOT, dry_run=False))
INDEX_PATH = GENERATION_DIR / "prepared_index.csv"
print("✓ данные готовы; использован существующий кеш:", cache_reused)

In [ ]:
index = validate_prepared_dataset(GENERATION_DIR, prep_config, full=True)
summary = index.group_by("split").agg(
    pl.col("cell_id").n_unique().alias("cells"), pl.col("rows").sum(), pl.col("windows").sum(),
).sort("split")
assert summary["windows"].min() > 0
print("✓ проверены все NPY, цели, окна и splits")
summary
DATASET_CELL_LIMIT = index.height
manifest = pl.read_csv(GENERATION_DIR / "manifest.csv")
raw_gb = manifest["size_bytes"].sum() / 1e9
prepared_gb = sum((GENERATION_DIR / p).stat().st_size for p in index["path"]) / 1e9
print(f"Физических батарей: {DATASET_CELL_LIMIT}; CSV: {raw_gb:.3f} ГБ")
print(f"Подготовленные NPY: {prepared_gb:.3f} ГБ; не размер VRAM для обучения")
print(summary)


In [ ]:
feature_scaler = json.loads((GENERATION_DIR / "scaler.json").read_text())
assert feature_scaler["feature"] == INPUT_COLUMNS
assert np.isfinite(feature_scaler["mean"]).all()
assert np.isfinite(feature_scaler["std"]).all() and (np.array(feature_scaler["std"]) > 0).all()
print("✓ scaler обучен на", feature_scaler["count"], "строках TRAIN")

### Проверка подготовленных сигналов
По одному непрерывному окну train/val/test. SOC/SOH в истории и будущие V/I/T показаны для проверки и не передаются модели. Затенение — горизонт прогноза; последний график — история SOH.

In [ ]:
def plot_prepared_data(split, seed=42):
    """Показать непрерывное окно и SOH одной батареи из split."""
    row = (
        index.filter(pl.col("split") == split)
        .sample(n=1, seed=seed).row(0, named=True)
    )
    data = np.load(GENERATION_DIR / row["path"], mmap_mode="r")
    offsets = json.loads(row["segment_offsets"])
    rng = np.random.default_rng(seed)
    segment = rng.integers(len(offsets) - 1)
    start, end = offsets[segment:segment + 2]
    length = INPUT_LENGTH + HORIZON
    count = (end - start - length) // WINDOW_STRIDE + 1
    start += WINDOW_STRIDE * int(rng.integers(count))

    values = data[start:start + length].astype(np.float64)
    mean = np.asarray(feature_scaler["mean"])
    std = np.asarray(feature_scaler["std"])
    values[:, :5] = values[:, :5] * std + mean
    minutes = (
        (np.arange(length) - INPUT_LENGTH + 1) * TARGET_INTERVAL_S / 60
    )

    fig, axes = plt.subplots(3, 2, figsize=(12, 8), layout="constrained")
    columns = [5, 6, 3, 2, 4]
    labels = ["SOC, %", "SOH, %", "Ток, А", "Напряжение, В", "Температура, °C"]
    for ax, col, label in zip(axes.flat, columns, labels):
        scale = 100 if col in (5, 6) else 1
        ax.plot(minutes, values[:, col] * scale, ".-")
        ax.axvline(0, color="gray", linestyle=":")
        ax.axvspan(0, minutes[-1], color="orange", alpha=0.12)
        ax.set(xlabel="Минуты относительно конца входа", ylabel=label)
        ax.grid(alpha=0.2)

    ids = np.linspace(0, len(data) - 1, min(2000, len(data)), dtype=int)
    days = (data[ids, 1].astype(float) - float(data[0, 1])) * std[1] / 86400
    axes[2, 1].scatter(days, 100 * data[ids, 6], s=5)
    axes[2, 1].set(xlabel="Дни с первой точки", ylabel="SOH, %")
    axes[2, 1].set_title("Доступная история батареи: выборка точек")
    axes[2, 1].grid(alpha=0.2)
    fig.suptitle(f'{split.upper()} | {row["cell_id"]}')
    plt.show()

In [ ]:
for split, seed in {"train": 101, "val": 202, "test": 303}.items():
    plot_prepared_data(split, seed)

## 3. Устройство и окна
Окна не сохраняются целиком: хранятся исходные строки и компактная таблица сегментов. Для текущего batch выделяется плотная память. Train перемешивает отдельные окна между батареями и возрастами; val/test имеют фиксированный порядок.
GPU-кеш включается, если строки всех split помещаются в выбранный лимит; иначе используется CPU mmap. Низкая занятость VRAM сама по себе не доказывает низкую скорость.

In [ ]:
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
precision = configure_precision(DEVICE, USE_MIXED_PRECISION,
                                amp_dtype=AMP_DTYPE, allow_tf32=ALLOW_TF32)
print("PyTorch:", torch.__version__, "Polars:", pl.__version__)
print("Устройство:", DEVICE, "AMP:", precision.dtype if precision.enabled else "выключен")
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(DEVICE))
    if torch.cuda.get_device_capability(DEVICE)[0] < 8:
        assert precision.dtype == torch.float16

In [ ]:
cache_device = choose_cache_device(GENERATION_DIR, DEVICE) if DATA_CACHE_DEVICE == "auto" else torch.device(DATA_CACHE_DEVICE)
window_config = WindowConfig(INPUT_LENGTH, HORIZON, WINDOW_STRIDE)
datasets = {
    split: WindowBatchDataset(GENERATION_DIR, split=split, config=window_config,
                              batch_size=8, shuffle=split == "train", seed=SEED, cache_device=cache_device, physics=PHYSICS_ENABLED)
    for split in ("train", "val", "test")
}
print("✓ кеш строк:", cache_device)
print("Окна:", {key: value.total_windows for key, value in datasets.items()})

In [ ]:
check_batch, check_meta = datasets["train"].fetch([0], metadata=True)
assert check_batch["x"].shape == (1, 32, 5)
assert check_batch["y_soc"].shape == check_batch["y_soh"].shape == (1, 8)
assert all(t.is_contiguous() and torch.isfinite(t).all() for t in check_batch.values())
source = int(check_meta["source"][0]); start = int(check_meta["start"][0])
source_path = GENERATION_DIR / datasets["train"].index.row(source, named=True)["path"]
source_array = np.load(source_path, mmap_mode="r")
np.testing.assert_allclose(check_batch["y_soc"][0].cpu(), source_array[start+32:start+40, 5])
np.testing.assert_allclose(check_batch["y_soh"][0].cpu(), source_array[start+32:start+40, 6])
print("✓ цели — строго следующие 8 точек, SOC/SOH не входят в x")
del source_array

In [ ]:
physics_target_check = {}
if physics_config is not None:
    diagnostic_batch = datasets["train"].sample_batch(min(1024, datasets["train"].total_windows))
    physics_target_check = diagnose_physics_targets(diagnostic_batch, physics_config)
    print("Согласованность TRAIN-меток с балансом заряда:", physics_target_check)
    print("Большая невязка требует проверки знака тока, единиц, SOC-коррекций и усреднения; вес loss не откалиброван.")
    del diagnostic_batch


## 4. Подбор batch
Probe измеряет полный train step на нескольких шагах и проверяет память/конечность. Среди близких по скорости размеров выбирается меньший. Результат зависит от GPU и форм входов; гарантировать максимум скорости без измерений нельзя.
Фиксированный batch задаётся только через `FIXED_MICRO_BATCH_SIZE`. Он тоже проверяется на GPU. На CPU перебор пропускается.

На CUDA сравниваются способы вычисления FEM и размеры batch, затем native scan с/без компиляции. Первый запуск может занять несколько минут. Время компиляции исключается из замера. Для resume используются сохранённые настройки. Автоподбор не использует val/test.

In [ ]:
if RESUME_PATH is not None:
    saved_setup = torch.load(RESUME_PATH, map_location="cpu", weights_only=True)
    assert saved_setup.get("version") == 3
    assert saved_setup.get("loss_config", {"sigma_floor": 1e-4, "physics": None}) == dict(sigma_floor=1e-4, **LOSS_CONFIG), "Смена loss требует нового эксперимента"
    MICRO_BATCH_SIZE = int(saved_setup["config"]["batch_size"])
    assert FIXED_MICRO_BATCH_SIZE in (None, MICRO_BATCH_SIZE)
    VRAM_TUNING_RESULT = saved_setup["config"].get("probe", {})
    EXECUTION = saved_setup["config"].get(
        "execution", {"fem_execution": "dynamic", "compile_scan": False}
    )
    del saved_setup
else:
    MICRO_BATCH_SIZE, VRAM_TUNING_RESULT = tune_training_setup(
        model_config, datasets["train"], precision,
        fixed=FIXED_MICRO_BATCH_SIZE, candidates=BATCH_SIZE_CANDIDATES,
        fem_execution=FEM_EXECUTION, compile_scan=COMPILE_SCAN,
        learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
        max_vram_fraction=MAX_VRAM_FRACTION,
        warmup_steps=PROBE_WARMUP_STEPS, timed_steps=PROBE_TIMED_STEPS, loss_config=LOSS_CONFIG,
    )
    EXECUTION = {k: VRAM_TUNING_RESULT[k]
                 for k in ("fem_execution", "compile_scan")}
print("Batch:", MICRO_BATCH_SIZE, "execution:", EXECUTION)
print("Измеренные настройки:", VRAM_TUNING_RESULT)


In [ ]:
for dataset in datasets.values():
    dataset.set_batch_size(MICRO_BATCH_SIZE)
batch_counts = {split: len(dataset) for split, dataset in datasets.items()}
assert all(value > 0 for value in batch_counts.values())
print("Число batch:", batch_counts)
print("Обычный effective batch:", MICRO_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)

## 5. Модель, loss и проверки
FEM: FFT по времени → top-k без DC → ceil(T/f) → padding/reshape → четыре последовательных блока `1×1 → grouped 3×3 → 1×1` → взвешенная сумма ветвей.
Эксперты SOC/shared/SOH содержат 2/4/6 FEM. Две оценки веса смешивают shared/task-признаки. SOC использует LSTM, SOH — Mamba по рис. 4. Глубокие признаки объединяются; `Conv1d(1) → ReLU → Conv1d(1)` выдаёт два горизонта по 8.

В loss sigma предсказываются из соответствующих Fw через Pool/Linear/Softplus. К MDEN loss добавлен отдельный мягкий баланс заряда по будущему току (только для loss) и штраф диапазона. Это экспериментальная физическая регуляризация, не точное воспроизведение loss статьи. Расчёт loss и рекуррентного состояния SSM выполняется в FP32. Softmax-веса FEM из амплитуд FFT и способ свёртки времени в голове — явно отмеченные допущения: точной спецификации авторы не дали.

In [ ]:
seed_everything(SEED)
model = MDEN(model_config).to(DEVICE)
EXECUTION = configure_execution(model, **EXECUTION)
print("✓ модель:", sum(p.numel() for p in model.parameters()), "параметров")
print("SSM backend:", model.soh_mamba.backend, "scan:", model.config.mamba_scan)

In [ ]:
criterion = MDENJointLoss(feature_dim=model_config.input_dim, **LOSS_CONFIG).to(DEVICE)
print("✓ отдельный обучаемый loss:", sum(p.numel() for p in criterion.parameters()), "параметров")

In [ ]:
smoke_batch = move_batch_to_device(datasets["train"].sample_batch(min(4, datasets["train"].total_windows)), DEVICE)
model.eval()
with torch.no_grad(), precision.context():
    smoke_output = model(smoke_batch["x"])
assert smoke_output["soc"].shape == smoke_batch["y_soc"].shape
assert smoke_output["soh"].shape == smoke_batch["y_soh"].shape
assert all(torch.isfinite(t).all() for t in smoke_output.values())
for key in ("soc_weights", "soh_weights"):
    torch.testing.assert_close(smoke_output[key].float().sum(-1), torch.ones(len(smoke_batch["x"]), device=DEVICE), atol=2e-3, rtol=2e-3)
print("✓ forward, формы, диапазоны весов и конечность")

In [ ]:
# FP32-проверка: прогноз того же окна не меняется от соседей в batch.
with torch.no_grad():
    together = model(smoke_batch["x"])
    alone = model(smoke_batch["x"][:1])
for task in ("soc", "soh"):
    torch.testing.assert_close(together[task][:1], alone[task], atol=3e-5, rtol=3e-4)
print("✓ независимость инференса от состава batch")

In [ ]:
model.train(); criterion.train()
with precision.context():
    smoke_output = model(smoke_batch["x"])
smoke_losses = criterion.from_batch(smoke_output, smoke_batch)
assert smoke_losses["loss"].dtype == torch.float32 and torch.isfinite(smoke_losses["loss"])
smoke_losses["loss"].backward()
all_parameters = list(model.parameters()) + list(criterion.parameters())
assert all(p.grad is None or torch.isfinite(p.grad).all() for p in all_parameters)
assert criterion.soc_uncertainty.linear.weight.grad is not None
assert criterion.soh_uncertainty.linear.weight.grad is not None
model.zero_grad(set_to_none=True); criterion.zero_grad(set_to_none=True)
print("✓ backward и градиенты обеих sigma-голов")
del smoke_output, smoke_losses, together, alone

## 6. Оптимизатор, scheduler и состояние — отдельные шаги
Joint loss остаётся целью обучения. `MONITOR` управляет scheduler/early stopping. Дополнительно сохраняются лучшие joint/SOC/SOH/balanced checkpoints по validation; test не используется для выбора модели.

In [ ]:
optimizer = build_optimizer(model, criterion, learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
assert sum(len(g["params"]) for g in optimizer.param_groups) == len(all_parameters)
print("✓ optimizer включает параметры модели и loss; A_log/D исключены из weight decay")

In [ ]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2, threshold=1e-4, min_lr=1e-5,
)
print("✓ scheduler создан; метрика:", MONITOR)

In [ ]:
grad_scaler = precision.scaler()
print("✓ GradScaler:", grad_scaler.is_enabled())

In [ ]:
run_name = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%f")
OUTPUT_DIR = Path(RESUME_PATH).resolve().parent if RESUME_PATH else PROJECT_ROOT / "runs" / f"mden_v3_{DATASET_CELL_LIMIT}cells_{run_name}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
save_data_recipe(GENERATION_DIR, OUTPUT_DIR)
(OUTPUT_DIR / "physics_target_check.json").write_text(json.dumps(physics_target_check, indent=2), encoding="utf-8")
training_state = new_training_state()
effective_config = dict(model=asdict(model_config), monitor=MONITOR, seed=SEED,
                        batch_size=MICRO_BATCH_SIZE, accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
                        learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY, epochs=NUM_EPOCHS,
                        patience=PATIENCE, gradient_clip_norm=GRADIENT_CLIP_NORM,
                        amp=str(precision.dtype) if precision.enabled else "off", probe=VRAM_TUNING_RESULT, execution=EXECUTION, allow_tf32=ALLOW_TF32)
print("✓ результаты:", OUTPUT_DIR)
(OUTPUT_DIR / "gpu_tuning.json").write_text(
    json.dumps(VRAM_TUNING_RESULT, indent=2, ensure_ascii=False), encoding="utf-8"
)


In [ ]:
if RESUME_PATH is not None:
    training_state = resume_training(RESUME_PATH, model, criterion, optimizer, scheduler,
                                     grad_scaler, effective_config, GENERATION_DIR)
    print("✓ продолжение после эпохи", training_state["epoch"])
else:
    seed_everything(SEED)
    print("✓ новое обучение")

## 7. Обучение
Одна внешняя полоса показывает эпохи; batch-полосы обновляются с `leave=False`, `mininterval=1`. В интерфейсе без поддержки обновления строк поставьте `SHOW_BATCH_PROGRESS=False`.
RMSE показан в процентных пунктах. Train-метрики собираются во время обновления весов и не равны отдельному eval-прогону train после эпохи.

In [ ]:
def progress_loader(dataset, description):
    return tqdm(dataset, total=len(dataset), desc=description, leave=False, position=1,
                mininterval=1.0, dynamic_ncols=True, disable=not SHOW_BATCH_PROGRESS)

In [ ]:
def run_one_epoch(epoch):
    started = time.perf_counter()
    datasets["train"].set_epoch(epoch)
    train_metrics = train_epoch_amp(
        model, criterion, progress_loader(datasets["train"], f"Train {epoch}"),
        optimizer, grad_scaler, precision, accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        gradient_clip_norm=GRADIENT_CLIP_NORM,
    )
    val_metrics = eval_epoch_amp(model, criterion, progress_loader(datasets["val"], "Validation"), precision)
    assert train_metrics["samples"] == datasets["train"].total_windows
    assert val_metrics["samples"] == datasets["val"].total_windows
    scheduler.step(val_metrics[MONITOR])
    return train_metrics, val_metrics, time.perf_counter() - started

In [ ]:
epochs = tqdm(range(training_state["epoch"] + 1, NUM_EPOCHS + 1), desc="Эпохи", position=0, disable=SMOKE_TEST)
for epoch in epochs:
    train_metrics, val_metrics, seconds = run_one_epoch(epoch)
    improved = complete_epoch(
        training_state, train_metrics, val_metrics, epoch=epoch,
        learning_rate=float(optimizer.param_groups[0]["lr"]), seconds=seconds, monitor=MONITOR,
    )
    save_epoch(OUTPUT_DIR, model, criterion, optimizer, scheduler, grad_scaler, training_state,
               config=effective_config, generation=GENERATION_DIR, improved=improved)
    epochs.set_postfix(val=f"{val_metrics['loss']:.4f}", SOC=f"{100*val_metrics['rmse_soc']:.2f} п.п.", SOH=f"{100*val_metrics['rmse_soh']:.2f} п.п.")
    tqdm.write(f"Epoch {epoch:02d}: train={train_metrics['loss']:.5f}; val={val_metrics['loss']:.5f}; "
               f"SOC={100*val_metrics['rmse_soc']:.3f} п.п.; SOH={100*val_metrics['rmse_soh']:.3f} п.п.; "
               f"{train_metrics['samples_per_second']:,.0f} samples/s; {seconds/60:.2f} мин; "
               f"AMP skipped={train_metrics['skipped_steps']}")
    if training_state["bad_epochs"] >= PATIENCE:
        tqdm.write("Early stopping по validation " + MONITOR)
        break
epochs.close()

In [ ]:
for name in TRACKED:
    assert (OUTPUT_DIR / f"best_{name}.pt").is_file()
assert (OUTPUT_DIR / "last.pt").is_file()
print("✓ лучшие эпохи:", training_state["best_epoch"])
print("✓ checkpoints и история сохранены")

## 8. Test выбранного заранее checkpoint
Используется критерий `MONITOR`, заданный до обучения. Мы не сравниваем test четырёх checkpoints для выбора лучшего. По каждому горизонту выводятся RMSE/MAE; дополнительные R²/WMAPE — по всем значениям. R² для постоянной цели не определён (`None`).

In [ ]:
SELECTED_CHECKPOINT = next(k for k, v in TRACKED.items() if v == MONITOR)
BEST_PATH = OUTPUT_DIR / f"best_{SELECTED_CHECKPOINT}.pt"
model, criterion, checkpoint = load_inference(BEST_PATH, DEVICE)
assert checkpoint["preprocessing"] == json.loads((GENERATION_DIR / "preprocessing.json").read_text())
print("✓ загружен:", BEST_PATH.name, "эпоха:", checkpoint["training_state"]["epoch"])

In [ ]:
test_metrics = eval_epoch_amp(model, criterion, progress_loader(datasets["test"], "Test"), precision)
assert test_metrics["samples"] == datasets["test"].total_windows
(OUTPUT_DIR / "test_metrics.json").write_text(json.dumps(test_metrics, indent=2), encoding="utf-8")
print("Test SOC RMSE:", 100 * test_metrics["rmse_soc"], "п.п.")
print("Test SOH RMSE:", 100 * test_metrics["rmse_soh"], "п.п.")

In [ ]:
horizon_table = pl.DataFrame({
    "horizon": np.arange(1, HORIZON + 1), "minutes": np.arange(1, HORIZON + 1) * TARGET_INTERVAL_S / 60,
    "soc_rmse_pp": np.array(test_metrics["rmse_soc_h"]) * 100,
    "soc_mae_pp": np.array(test_metrics["mae_soc_h"]) * 100,
    "soh_rmse_pp": np.array(test_metrics["rmse_soh_h"]) * 100,
    "soh_mae_pp": np.array(test_metrics["mae_soh_h"]) * 100,
})
horizon_table.write_csv(OUTPUT_DIR / "test_horizons.csv")
horizon_table

## 9. История и шесть примеров
В каждом split выбираются две разные батареи (если в нём есть минимум две). Выбор фиксируется seed и не зависит от ошибки модели. Синие исторические SOC/SOH служат только контекстом: они не передаются модели. Прогнозы не ограничиваются искусственно диапазоном и не сглаживаются перед оценкой.

In [ ]:
history_figure = plot_history(training_state["history"], OUTPUT_DIR)

In [ ]:
prediction_examples = {}
for split, seed in {"train": 101, "val": 202, "test": 303}.items():
    examples = collect_examples(model, datasets[split], precision, count=2, seed=seed)
    prediction_examples[split] = examples
    plot_examples(examples, interval_s=TARGET_INTERVAL_S, output_dir=OUTPUT_DIR)
print("✓ прогнозы и целевые значения сохранены также в CSV")

## 10. Что означает результат
- 32 блока × 30 секунд = 16 минут наблюдений; следующие 8 блоков = 4 минуты прогноза. Между первой и последней отметкой входного ряда — 15.5 минуты.
- Модель не получает будущие V/I/T. При неизвестной будущей нагрузке прогноз ограничен режимами, выученными из данных.
- SOH здесь обучается на интерполированной оценке ёмкости, а не на независимом измерении каждые 30 секунд.
- Разные лучшие эпохи SOC/SOH не доказывают конфликт градиентов. Возрастающий val при падающем train совместим с переобучением, но также требует проверки распределений и взвешивания loss.
- Метрики MIT из статьи нельзя считать ожидаемой точностью KIT. Точные гиперпараметры, FFT-агрегация и временная редукция prediction head не раскрыты полностью.

Полная карта изменений и ограничений — `docs/AUDIT_RU.md`; последовательная схема кода — `docs/PIPELINE_RU.md`.

## Очистка пересоздаваемых данных
Выполнить после всех графиков. NPY удаляются, checkpoints и data_recipe сохраняются. Для повторной оценки/обучения нужно заново выполнить подготовку и создать datasets. Остановите старые процессы предыдущих версий, которые не используют блокировку кеша.

In [ ]:
if DELETE_PREPARED_AFTER_RUN:
    for dataset in datasets.values():
        dataset.close()
    cleanup_report = cleanup_prepared_cache(PREPARED_ROOT, include_current=True, dry_run=False)
    print("Очистка:", cleanup_report)
    (OUTPUT_DIR / "cache_cleanup.json").write_text(json.dumps(cleanup_report, indent=2), encoding="utf-8")
